# 🧩 Mini-Lab: Fake AI Client Tests

**Module: Tests That Catch Real Bugs** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** what a test double is and why it replaces a real AI client in tests
2. **Apply** dependency injection to make functions testable without network access
3. **Verify** that retry logic and error-handling paths work correctly using fake clients that simulate failures
4. **Identify** how output-parsing tests can use fake responses to cover malformed JSON edge cases

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Test Double | An injected fake backend that can return malformed JSON or time out on demand, so tests reliably exercise error paths without calling real external services. |
| Dependency Injection | Pass the backend in, callers choose which model client to use, while your pipeline code stays unchanged and testable. |
| Bounded Retry | Retry transient errors only, and cap attempts, total wall-clock time, and cost or rate-limit budget so failures end fast, predictably, and safely. |
| Output Parsing | Output parsing is turning a model’s response into reliable, structured data (like JSON fields) your code can safely use,while detecting mistakes, missing info, or prompt-injection tricks in the output. |


## The Problem: Testing Code That Calls an LLM

The LLM call is stochastic, but the **code around it** (retries, error handling, JSON parsing) is deterministic. We can test that code offline and for free by **injecting a fake AI client** that returns valid JSON or malformed JSON, or raises an exception, on demand. First, here is the hard-coded version that cannot be tested:

In [1]:
# Skip if already installed
!pip install -q pytest ipytest


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\molak\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import ipytest
ipytest.autoconfig()

In [3]:
import json


# ❌ Hard-coded dependency: the client would be created inside, so tests can't swap it out
def extract_contact_hardcoded(text: str) -> dict:
    # client = OpenAI()  ← needs network + API key
    raise NotImplementedError("Can't test this without a real API key!")

## Step 1 — Dependency Injection Makes It Testable

We pass the AI client **as a parameter**, so the function works with a real OpenAI client or a fake. Then we use a fake to test the function's bounded retry and JSON parsing.

In [4]:
import json
import time


# client: any object with a .chat.completions.create() method (injected)
def extract_contact(text: str, client, max_retries: int = 2) -> dict:
    messages = [
        {"role": "system", "content": "Extract name and email as JSON."},
        {"role": "user", "content": text},
    ]

    last_error = None
    for attempt in range(max_retries + 1):  # bounded retry
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
            )
            raw = response.choices[0].message.content
            return json.loads(raw)  # output parsing
        except json.JSONDecodeError as e:
            # Bad output is not transient, so fail right away without retrying
            raise ValueError(f"Malformed JSON from LLM: {e}") from e
        except Exception as e:
            last_error = e
            if attempt < max_retries:
                time.sleep(0)

    raise RuntimeError(
        f"All {max_retries + 1} attempts failed. Last error: {last_error}"
    )

## Step 2 — Building a Fake AI Client (Test Double)

> 🧩 **Fake backend in tests.** This is the injected fake from the video, now built in code.

Our fake mimics the real client's interface. It needs only `client.chat.completions.create()`, and we control what it returns or raises.

In [5]:
# Nested objects so response.choices[0].message.content works
class FakeMessage:
    def __init__(self, content: str):
        self.content = content

class FakeChoice:
    def __init__(self, content: str):
        self.message = FakeMessage(content)

class FakeResponse:
    def __init__(self, content: str):
        self.choices = [FakeChoice(content)]


class FakeAIClient:
    # responses: strings returned in order; error: raised on every call
    def __init__(self, responses=None, error=None):
        self.responses = list(responses) if responses else []
        self.error = error
        self.call_count = 0
        self.chat = self  # so client.chat.completions.create() works
        self.completions = self

    def create(self, **kwargs):
        self.call_count += 1
        if self.error:
            raise self.error
        return FakeResponse(self.responses.pop(0))

## Step 3 — Testing the Happy Path

First, let's confirm that when the fake client returns valid JSON, our function parses it correctly.

In [6]:
%%ipytest -v

import pytest


def test_happy_path_returns_parsed_dict():
    """Valid JSON response → parsed dict."""
    fake = FakeAIClient(
        responses=['{"name": "Alice", "email": "alice@example.com"}']
    )
    result = extract_contact("Contact Alice at alice@example.com", client=fake)

    assert result == {"name": "Alice", "email": "alice@example.com"}
    assert fake.call_count == 1  # called exactly once, no retries needed

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\mini-fake-ai-client-tests
plugins: anyio-4.15.1, langsmith-0.14.1, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 1 item

t_7ed7f8a4189a43188fb87faa9041922e.py 

.

                                                      [100%]



======================================== 1 passed in 0.07s ========================================


## Step 4 — Testing Malformed JSON (Output Parsing Failure)

What happens when the LLM returns something that isn't valid JSON? Our function should raise a `ValueError`. Let's prove it.

In [7]:
%%ipytest -v


def test_malformed_json_raises_value_error():
    """Non-JSON response → ValueError."""
    fake = FakeAIClient(responses=["Sorry, I can't extract that."])

    with pytest.raises(ValueError, match="Malformed JSON"):
        extract_contact("some text", client=fake)

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\mini-fake-ai-client-tests
plugins: anyio-4.15.1, langsmith-0.14.1, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 1 item

t_7ed7f8a4189a43188fb87faa9041922e.py 

.

                                                      [100%]



======================================== 1 passed in 0.01s ========================================


With a fake, we can force malformed JSON every time and test this path deterministically.

## Step 5 — Testing Bounded Retry on API Failures

Now the most valuable test: does the retry logic actually work? We simulate an API that always times out.

In [8]:
%%ipytest -v


def test_retries_exhausted_raises_runtime_error():
    """All retries fail → RuntimeError with attempt count."""
    fake = FakeAIClient(error=TimeoutError("Connection timed out"))

    with pytest.raises(RuntimeError, match="All 3 attempts failed"):
        extract_contact("some text", client=fake, max_retries=2)

    # 1 initial attempt + 2 retries = 3 total calls
    assert fake.call_count == 3

======================================= test session starts =======================================


platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\mini-fake-ai-client-tests
plugins: anyio-4.15.1, langsmith-0.14.1, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 1 item

t_7ed7f8a4189a43188fb87faa9041922e.py 

.

                                                      [100%]



======================================== 1 passed in 0.02s ========================================


## 🎯 Summary

### Key Takeaways

1. **Test doubles replace real dependencies** — a simple `FakeAIClient` class mimics the OpenAI interface and returns controlled responses, making tests fast, free, and deterministic
2. **Dependency injection is the enabler** — passing the client as a parameter (instead of creating it inside the function) is the single change that makes offline testing possible
3. **Failure paths are the most valuable tests** — malformed JSON, timeouts, and exhausted retries are hard to trigger with a real API but trivial to simulate with a fake
4. **Bounded retry is verifiable** — by checking `fake.call_count`, you can assert that the retry loop ran exactly the right number of times